# Setting

## Import Packages

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time

pd.set_option('display.precision', 4)

## Self-defined functions

### check_table_info

In [2]:
def check_table_info(target_df):
    """
    To check the unique values, dtype, example, missing rate of each column
    """
    table_info = []
    for col in target_df:
        table_info_row = []
        table_info_row.append(col)
        table_info_row.append(target_df[col].nunique())
        table_info_row.append(target_df[col].dtype)
        table_info_row.append(target_df[col].iloc[0])
        table_info_row.append(round(target_df[col].isna().sum() / target_df.shape[0]*100,2))

        table_info.append(table_info_row)
    res = pd.DataFrame(table_info, columns=['col_name', 'unique_values', 'dtype', 'example', 'missing%'])

    return res

# NCEI Hourly Weather Data
- Data Source: [National Centers for Environmental Information](https://www.ncei.noaa.gov/access/search/data-search/global-hourly?bbox=40.839,-74.090,40.667,-73.918&pageNum=1&startDate=2024-06-01T00:00:00&endDate=2025-05-31T23:59:59)
  - There are three stations for Manhattan: NY CITY CENTRAL PARK, PORT AUTH DOWNTN MANHATTAN WALL ST HEL, and THE BATTERY
  - Given Manhattan’s relatively small geographic area, the variation in observations between stations is minimal. Therefore, we use data from **NY CITY CENTRAL PARK** as a representative source for hourly weather conditions in Manhattan.
- Date Range: 2024/01/01 - 2025/05/31

In [3]:
weather_2024 = pd.read_csv('raw_data/hourly_weather_2024.csv')
weather_2025 = pd.read_csv('raw_data/hourly_weather_20250602.csv')
hourly_weather_data = pd.concat([weather_2024, weather_2025])

hourly_weather_data['date'] = pd.to_datetime(pd.to_datetime(hourly_weather_data['DATE']).dt.date)
hourly_weather_data['hour'] = pd.to_datetime(hourly_weather_data['DATE']).dt.hour
hourly_weather_data["temp_c"] = hourly_weather_data["TMP"].str.split(",").str[0].astype(float) / 10
hourly_weather_data["dew_c"] = hourly_weather_data["DEW"].str.split(",").str[0].astype(float) / 10
hourly_weather_data["wind_speed_knot"] = hourly_weather_data["WND"].str.split(",").str[3].astype(float) / 10
hourly_weather_data["precip_mm"] = hourly_weather_data["AA1"].str.split(",").str[1].astype(float) / 10

weather_df = hourly_weather_data[['DATE', 'date', 'hour', 'temp_c', 'dew_c', 'wind_speed_knot', 'precip_mm']].copy()
weather_df = weather_df.sort_values('DATE')

/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/ipykernel_61097/880568346.py:1: DtypeWarning: Columns (36,43,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  weather_2024 = pd.read_csv('raw_data/hourly_weather_2024.csv')


In [4]:
# Only keep data between 2024-04-01 and 2025-03-31
start_date = pd.to_datetime('2024-04-01')
end_date = pd.to_datetime('2025-04-01')  # exclusive upper bound
weather_df = weather_df[weather_df['date'].between(start_date, end_date - pd.Timedelta(seconds=1))]

In [13]:
# Replace invalid data with null
weather_df = weather_df.replace(999.9, np.nan)

# Drop the record if the temp_c column is missing
weather_df = weather_df[weather_df['temp_c'].notna()]

# Use the last record to represent that hour
weather_df = weather_df.groupby(['date', 'hour']).tail(1).reset_index(drop=True)

# Use the date_hour_df to ensure every hour has corresponding weather data
dates = pd.date_range(start=start_date.date(), end=(end_date - pd.Timedelta(days=1)).date(), freq="D").date
hours = list(range(24))
date_hour = list(itertools.product(dates, hours))
date_hour_df = pd.DataFrame(date_hour, columns=["date", "hour"])
date_hour_df['date'] = pd.to_datetime(date_hour_df['date'])
final_weather_df = date_hour_df.merge(weather_df, on=['date', 'hour'], how='left')

# Drop unnecessary columns
final_weather_df.drop(columns='DATE', inplace=True)

# Use the previous record to fill the missing value
final_weather_df = final_weather_df.ffill()
check_table_info(final_weather_df)

,col_name,unique_values,dtype,example,missing%
0,date,365,datetime64[ns],2024-04-01 00:00:00,0.0
1,hour,24,int64,0,0.0
2,temp_c,105,float64,13.3,0.0
3,dew_c,106,float64,-0.6,0.0
4,wind_speed_knot,20,float64,0.0,0.0
5,precip_mm,40,float64,0.0,0.0


In [14]:
# No missing values
check_table_info(final_weather_df)

,col_name,unique_values,dtype,example,missing%
0,date,365,datetime64[ns],2024-04-01 00:00:00,0.0
1,hour,24,int64,0,0.0
2,temp_c,105,float64,13.3,0.0
3,dew_c,106,float64,-0.6,0.0
4,wind_speed_knot,20,float64,0.0,0.0
5,precip_mm,40,float64,0.0,0.0


In [17]:
final_weather_df.to_csv('hourly_weather_data_20240401_20250331.csv', index=False)

In [18]:
final_weather_df

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm
0,2024-04-01,0,13.3,-0.6,0.0,0.0
1,2024-04-01,1,13.3,-0.6,1.5,0.0
2,2024-04-01,2,12.8,-0.6,0.0,0.0
3,2024-04-01,3,12.2,-0.6,0.0,0.0
4,2024-04-01,4,12.2,-1.1,0.0,0.0
...,...,...,...,...,...,...
8755,2025-03-31,19,21.1,12.2,2.6,0.0
8756,2025-03-31,20,21.7,11.7,3.1,0.0
8757,2025-03-31,21,20.0,14.4,2.6,0.5
8758,2025-03-31,22,16.7,15.0,1.5,0.0
